<a href="https://colab.research.google.com/github/dhruvpesi2026-ui/PredictionModelDhruv/blob/main/heartDisease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier

In [2]:
# CSV load
df = pd.read_csv('heart_disease.csv')

# Basic cleaning
df.columns = df.columns.str.strip()

# Data check
print(df.head())
print(df.shape)
print(df.dtypes)

   age     sex  cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0   42    male   0      94.0  310.5    1        0    179.4      0     1.85   
1   31  Female   2     116.9  248.6    0        0    153.5      1     1.99   
2   64    Male   3     133.3  260.9    0        1    136.8      0     0.07   
3   58    male   0     124.4  229.7    0        0    173.9      0     1.68   
4   45  Female   1     148.8  258.7    0        0    186.3      1     1.16   

   slope  ca  thal  target  
0      2   2     1       0  
1      1   0     2       0  
2      2   0     2       1  
3      1   2     1       0  
4      1   2     2       0  
(500, 14)
age           int64
sex          object
cp            int64
trestbps    float64
chol        float64
fbs           int64
restecg       int64
thalach     float64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object


In [3]:
# Target split
X = df.drop('target', axis=1)
y = df['target']

# Gender/sex column cleanup
if 'sex' in X.columns:
    X['sex'] = X['sex'].astype(str).str.strip().str.lower()

print(X.head())
print(y.head())

   age     sex  cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0   42    male   0      94.0  310.5    1        0    179.4      0     1.85   
1   31  female   2     116.9  248.6    0        0    153.5      1     1.99   
2   64    male   3     133.3  260.9    0        1    136.8      0     0.07   
3   58    male   0     124.4  229.7    0        0    173.9      0     1.68   
4   45  female   1     148.8  258.7    0        0    186.3      1     1.16   

   slope  ca  thal  
0      2   2     1  
1      1   0     2  
2      2   0     2  
3      1   2     1  
4      1   2     2  
0    0
1    0
2    1
3    0
4    0
Name: target, dtype: int64


In [4]:
# Identify column types
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['age', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Categorical columns: ['sex']


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numeric data ke liye
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical data ke liye
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Dono ko ek ColumnTransformer me jodo
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

In [7]:
from sklearn.pipeline import Pipeline

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(400, 13) (100, 13)


In [9]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'cp', 'trestbps',
                                                   'chol', 'fbs', 'restecg',
                                                   'thalach', 'exang',
                                                   'oldpeak', 'slope', 'ca',
                                                   'thal']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex'])])),
                ('model',
                 RandomForestClassifier(max_depth=12, min_samples_leaf=2,
                                        min_samples_split=5, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc * 100, 2), "%")

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 67.0 %

Confusion Matrix:
 [[45 15]
 [18 22]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.75      0.73        60
           1       0.59      0.55      0.57        40

    accuracy                           0.67       100
   macro avg       0.65      0.65      0.65       100
weighted avg       0.67      0.67      0.67       100

